# 3. Loan default machine-learning models

Train and compare Logistic Regression, Random Forest, and XGBoost using a stratified sample of up to 50,000 records. The split and preprocessing follow the project pipeline: preprocessing is fitted only on training data, model selection uses validation PR-AUC, and the test split is reserved for final evaluation. This notebook is a compact learning workflow; the full project training pipeline is available with `python -m src.train`.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data" / "processed").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
CLEAN_PATH = PROJECT_ROOT / "data" / "processed" / "loan_default_cleaned.csv"
if not CLEAN_PATH.exists():
    raise FileNotFoundError("Run notebooks/01_Data_Cleaning.ipynb first to create the cleaned dataset.")

from src.config import CATEGORICAL_COLUMNS, ENGINEERED_COLUMNS, ID_COLUMNS, NUMERIC_COLUMNS, RANDOM_STATE, TARGET
from src.preprocessing import add_features, build_preprocessor

df = pd.read_csv(CLEAN_PATH)
print(f"Loaded {len(df):,} cleaned loans")

## Create a manageable stratified modeling sample

The sample is capped at 50,000 rows to keep notebook training practical. It retains the dataset's default/non-default balance.

In [ ]:
MAX_SAMPLE_SIZE = 50_000
if len(df) > MAX_SAMPLE_SIZE:
    model_df, _ = train_test_split(
        df, train_size=MAX_SAMPLE_SIZE, stratify=df[TARGET], random_state=RANDOM_STATE
    )
else:
    model_df = df.copy()

model_df = add_features(model_df).drop(columns=ID_COLUMNS, errors="ignore")
feature_columns = NUMERIC_COLUMNS + ENGINEERED_COLUMNS + CATEGORICAL_COLUMNS
X, y = model_df[feature_columns], model_df[TARGET]

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.40, stratify=y, random_state=RANDOM_STATE
)
X_validation, X_test, y_validation, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=RANDOM_STATE
)
print(f"Train: {len(X_train):,} | validation: {len(X_validation):,} | test: {len(X_test):,}")
print(f"Default rate in train: {y_train.mean():.2%}; validation: {y_validation.mean():.2%}; test: {y_test.mean():.2%}")

## Define and train models

Numeric and categorical preprocessing is included within each model pipeline to prevent information leakage from validation/test data. Class weights help address the minority default class.

In [ ]:
negative_to_positive = float((y_train == 0).sum() / (y_train == 1).sum())
classifiers = {
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight="balanced"),
    "Random Forest": RandomForestClassifier(
        n_estimators=100, class_weight="balanced", min_samples_leaf=20,
        n_jobs=-1, random_state=RANDOM_STATE,
    ),
    "XGBoost": XGBClassifier(
        n_estimators=100, max_depth=4, learning_rate=0.1,
        scale_pos_weight=negative_to_positive, tree_method="hist",
        eval_metric="logloss", n_jobs=-1, random_state=RANDOM_STATE,
    ),
}

fitted_models = {}
validation_scores = {}
for name, classifier in classifiers.items():
    pipeline = Pipeline([
        ("prep", build_preprocessor()),
        ("model", classifier),
    ])
    pipeline.fit(X_train, y_train)
    fitted_models[name] = pipeline
    validation_scores[name] = pipeline.predict_proba(X_validation)[:, 1]
    print(f"{name} trained")

## Select on validation data and evaluate once on the held-out test data

PR-AUC is used to select the model because defaults are the minority class. The table reports test-set precision and recall at a fixed 0.50 threshold; the production workflow additionally selects a threshold on validation data.

In [ ]:
selected_model = max(
    validation_scores,
    key=lambda name: average_precision_score(y_validation, validation_scores[name]),
)
results = []
for name, pipeline in fitted_models.items():
    probabilities = pipeline.predict_proba(X_test)[:, 1]
    predictions = probabilities >= 0.5
    results.append({
        "Model": name,
        "Validation PR-AUC": average_precision_score(y_validation, validation_scores[name]),
        "Test PR-AUC": average_precision_score(y_test, probabilities),
        "Test ROC-AUC": roc_auc_score(y_test, probabilities),
        "Test precision (0.50 threshold)": precision_score(y_test, predictions, zero_division=0),
        "Test recall (0.50 threshold)": recall_score(y_test, predictions, zero_division=0),
    })

results = pd.DataFrame(results).sort_values("Validation PR-AUC", ascending=False).reset_index(drop=True)
print(f"Selected model by validation PR-AUC: {selected_model}")
display(results.round(3))